# 🧪 Gemastik 2026: Pretrained Base Model Zero-Shot Multimodal Evaluation
### 🔍 Zero-Shot Inference with Corrected Pretrained `id2label` Mapping

This notebook evaluates the **raw pretrained base models** out-of-the-box (zero-shot, without fine-tuning) on the Indonesian video dataset (`val_split.json`):
1. 📹 **Video Base Model:** `dima806/deepfake_vs_real_image_detection` (Vision Transformer - ViT)
2. 🎙️ **Audio Base Model:** `MelodyMachine/Deepfake-audio-detection-V2` (Wav2Vec2)
3. 🤝 **Raw Multimodal Ensemble:** Late fusion of un-finetuned visual and acoustic models ($0.5 \cdot P_{\text{video}} + 0.5 \cdot P_{\text{audio}}$)

---
### ⚠️ Critical `id2label` Alignment Notice
- **Video Model (`dima806`):** HuggingFace config `id2label` = `{"0": "Real", "1": "Fake"}` $\rightarrow$ Matches dataset (`0 = Real, 1 = AI`).
- **Audio Model (`MelodyMachine`):** HuggingFace config `id2label` = `{"0": "fake", "1": "real"}` $\rightarrow$ **INVERTED** relative to dataset (`0 = Real, 1 = AI`).
- **Correction:** Output index `0` ("fake") is mapped to `1` (AI) and index `1` ("real") is mapped to `0` (Real) during zero-shot audio evaluation. This resolves the previous 33.65% inverted baseline artifact and restores true zero-shot audio accuracy (~66.35%).

## 1. Setup Environment & Mount Google Drive

In [ ]:
# Install required audio & vision libraries
!pip install -q transformers datasets torchaudio librosa soundfile scikit-learn matplotlib seaborn tqdm pandas opencv-python-headless pillow

import os
import glob
import shutil
import zipfile
import subprocess
import json
from pathlib import Path
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import soundfile as sf
import torch
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoFeatureExtractor,
    AutoModelForAudioClassification
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Running inference on device: {device}")

# Optional: Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("ℹ️ Local or Non-Colab environment detected.")

## 2. Pretrained Model Backbone & Directory Setup

In [ ]:
# Pretrained Model IDs on Hugging Face Hub
VIDEO_BASE_MODEL_ID = "dima806/deepfake_vs_real_image_detection"
AUDIO_BASE_MODEL_ID = "MelodyMachine/Deepfake-audio-detection-V2"

CLASSES = ["Real", "AI"]
IMAGE_SIZE = (224, 224)
SAMPLE_RATE = 16000
MAX_AUDIO_SAMPLES = int(5.0 * SAMPLE_RATE)  # 5-second chunk window

# Inspect Native id2label Configs from Hugging Face Hub
cfg_vid = AutoConfig.from_pretrained(VIDEO_BASE_MODEL_ID)
cfg_aud = AutoConfig.from_pretrained(AUDIO_BASE_MODEL_ID)

print("==========================================")
print(f"📹 Video Base Model: {VIDEO_BASE_MODEL_ID}")
print(f"   Native id2label: {cfg_vid.id2label}")
print("==========================================")
print(f"🎙️ Audio Base Model: {AUDIO_BASE_MODEL_ID}")
print(f"   Native id2label: {cfg_aud.id2label}")
print("==========================================")

# Directory Paths (Isolated Checkpoint Save Path)
DRIVE_DATASET_DIR = "/content/drive/MyDrive/Gemastik26/Dataset Indonesia"
KFOLD_SPLIT_FILE = "/content/drive/MyDrive/Gemastik26/kfold_splits.json"
MODEL_SAVE_DIR = "/content/drive/MyDrive/Gemastik26/models/revision1"

ZIP_VAL_PATHS = {
    "Real": os.path.join(DRIVE_DATASET_DIR, "Real", "train.zip"),
    "AI": os.path.join(DRIVE_DATASET_DIR, "AI", "trainAI.zip")
}

LOCAL_RAW_VAL_DIR = "/content/dataset_raw_val"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


## 3. Unzip Video Archives to Colab SSD (Smart Skip)

In [ ]:
def unzip_file(zip_path, extract_to):
    if not os.path.exists(zip_path):
        print(f"⚠️ Warning: Zip file not found at {zip_path}")
        return False
    print(f"📦 Unzipping {zip_path} -> {extract_to} ...")
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✅ Unzipped successfully to {extract_to}")
    return True

for cls in CLASSES:
    extract_target = os.path.join(LOCAL_RAW_VAL_DIR, cls)
    zip_p = ZIP_VAL_PATHS.get(cls, "")
    if os.path.exists(extract_target) and len(os.listdir(extract_target)) > 0:
        print(f"⏭️ Target directory '{extract_target}' already exists. Skipping unzip.")
    else:
        unzip_file(zip_p, extract_target)

print("✅ Validation dataset ready on Colab SSD!")

## 4. Video Frame & Audio Chunk Extraction Helpers

In [ ]:
def extract_frames_from_video(video_path, target_fps=1.0, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0 or np.isnan(video_fps):
        video_fps = 30.0

    frame_interval = max(1, int(round(video_fps / target_fps)))
    frames = []
    count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_interval == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            frames.append(img)
            if len(frames) >= max_frames:
                break
        count += 1

    cap.release()
    return frames

def extract_audio_chunks(video_path, temp_dir="/content/temp_base_audio", chunk_sec=5.0):
    os.makedirs(temp_dir, exist_ok=True)
    temp_wav = os.path.join(temp_dir, "_temp.wav")

    cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", str(SAMPLE_RATE), "-ac", "1",
        temp_wav
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if not os.path.exists(temp_wav) or os.path.getsize(temp_wav) < 1000:
        return []

    y, sr = sf.read(temp_wav)
    os.remove(temp_wav)

    chunk_samples = int(chunk_sec * sr)
    total_samples = len(y)
    if total_samples == 0:
        return []

    num_chunks = max(1, int(np.ceil(total_samples / chunk_samples)))
    chunks = []
    for i in range(num_chunks):
        start = i * chunk_samples
        end = min(total_samples, (i + 1) * chunk_samples)
        c_data = y[start:end]
        if len(c_data) < int(1.0 * sr):
            continue
        if len(c_data) < chunk_samples:
            c_data = np.pad(c_data, (0, chunk_samples - len(c_data)), mode="constant")
        chunks.append(c_data)
    return chunks

## 5. Load Validation Dataset Split (`val_split.json`)

In [ ]:
val_videos = []

# Load Fold-1 validation stems from the revision pipeline's kfold_splits.json
val_stems = None
base_fold_name = None
if os.path.exists(KFOLD_SPLIT_FILE):
    with open(KFOLD_SPLIT_FILE, "r", encoding="utf-8") as f:
        split_data = json.load(f)
    base_fold_name = list(split_data.keys())[0]
    val_stems = set(split_data[base_fold_name].get("val_stems", []))
    print(f"✅ Loaded {len(val_stems)} validation video stems from {KFOLD_SPLIT_FILE} (fold: {base_fold_name})")
else:
    print("⚠️ kfold_splits.json not found - evaluating the FULL train dataset as 'Full Dataset Zero-Shot Baseline'.")

for label_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(LOCAL_RAW_VAL_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    video_extensions = ("*.mp4", "*.avi", "*.mov", "*.mkv", "*.MP4", "*.AVI", "*.MOV", "*.MKV")
    vfiles = []
    for ext in video_extensions:
        vfiles.extend(glob.glob(os.path.join(cls_dir, "**", ext), recursive=True))
    
    for vf in vfiles:
        stem = Path(vf).stem
        if val_stems is not None and stem not in val_stems:
            continue  # Keep only validation split videos
        val_videos.append((vf, label_idx, cls, stem))

print(f"📊 Total Validation Videos to Evaluate: {len(val_videos)}")

## 6. Base Model Inference Engine with Explicit `id2label` Remapping

In [ ]:
print("⏳ Loading Pretrained Video ViT Base Model ...")
video_processor = AutoImageProcessor.from_pretrained(VIDEO_BASE_MODEL_ID)
video_base_model = AutoModelForImageClassification.from_pretrained(VIDEO_BASE_MODEL_ID).to(device)
video_base_model.eval()

print("⏳ Loading Pretrained Audio Wav2Vec2 Base Model ...")
audio_feature_extractor = AutoFeatureExtractor.from_pretrained(AUDIO_BASE_MODEL_ID)
audio_base_model = AutoModelForAudioClassification.from_pretrained(AUDIO_BASE_MODEL_ID).to(device)
audio_base_model.eval()

print("✅ Raw Pretrained Base Models Loaded Successfully!")

video_val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=video_processor.image_mean, std=video_processor.image_std)
])

def predict_video_base_probs(video_path):
    """
    Predicts raw video probabilities using dima806 ViT base model.
    dima806 id2label: {0: 'Real', 1: 'Fake'}
    Returns [p_real, p_ai]
    """
    frames = extract_frames_from_video(video_path, target_fps=1.0, max_frames=30)
    if not frames:
        return np.array([0.5, 0.5])
    
    tensors = torch.stack([video_val_transform(f) for f in frames]).to(device)
    with torch.no_grad():
        outputs = video_base_model(tensors)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        avg_prob = np.mean(probs, axis=0)
    
    # Native mapping: index 0 = Real, index 1 = Fake (AI)
    return avg_prob

def predict_audio_base_probs(video_path):
    """
    Predicts raw audio probabilities using MelodyMachine Wav2Vec2 base model.
    MelodyMachine id2label: {0: 'fake', 1: 'real'}
    EXPLICIT REMAPPING to dataset convention [0 = Real, 1 = AI]:
    p_real = prob_raw[1] (real)
    p_ai   = prob_raw[0] (fake)
    """
    chunks = extract_audio_chunks(video_path, chunk_sec=5.0)
    if not chunks:
        return np.array([0.5, 0.5])
    
    chunk_probs = []
    with torch.no_grad():
        for cdata in chunks:
            inputs = audio_feature_extractor(cdata, sampling_rate=SAMPLE_RATE, return_tensors="pt")
            input_values = inputs.input_values.to(device)
            outputs = audio_base_model(input_values)
            probs_raw = F.softmax(outputs.logits, dim=-1).squeeze(0).cpu().numpy()
            
            # Explicit Remapping:
            # MelodyMachine index 0 = fake, index 1 = real
            # Dataset index 0 = Real, index 1 = AI (fake)
            p_real = float(probs_raw[1])
            p_ai   = float(probs_raw[0])
            chunk_probs.append([p_real, p_ai])
    
    avg_prob = np.mean(chunk_probs, axis=0)
    return avg_prob

## 7. Run Zero-Shot Base Model Multimodal Evaluation

In [ ]:
val_targets = []
val_video_base_preds = []
val_audio_base_preds = []
val_fusion_base_preds = []

WEIGHT_VIDEO = 0.5
WEIGHT_AUDIO = 0.5

print(f"🚀 Evaluating {len(val_videos)} Validation Videos on Raw Base Models ...")

for vfile, label, cls_name, vid_stem in tqdm(val_videos, desc="Base Model Zero-Shot Eval"):
    val_targets.append(label)
    
    p_vid = predict_video_base_probs(vfile)
    p_aud = predict_audio_base_probs(vfile)
    p_fusion = (WEIGHT_VIDEO * p_vid) + (WEIGHT_AUDIO * p_aud)
    
    val_video_base_preds.append(int(np.argmax(p_vid)))
    val_audio_base_preds.append(int(np.argmax(p_aud)))
    val_fusion_base_preds.append(int(np.argmax(p_fusion)))

print("✅ Zero-Shot Base Model Evaluation Complete!")

## 8. Quantitative Baseline Metrics & Confusion Matrices

In [ ]:
def compute_metrics(y_true, y_pred, name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "Model": name,
        "Accuracy": f"{acc * 100:.2f}%",
        "Precision (Macro)": f"{prec * 100:.2f}%",
        "Recall (Macro)": f"{rec * 100:.2f}%",
        "F1-Score (Macro)": f"{f1 * 100:.2f}%"
    }

results = [
    compute_metrics(val_targets, val_video_base_preds, "Raw Video ViT (dima806)"),
    compute_metrics(val_targets, val_audio_base_preds, "Raw Audio Wav2Vec2 (MelodyMachine - Corrected Mapping)"),
    compute_metrics(val_targets, val_fusion_base_preds, "Raw Multimodal Ensemble (Late Fusion)")
]

df_metrics = pd.DataFrame(results)
print("========================================================")
print("📊 PRETRAINED BASE MODEL ZERO-SHOT EVALUATION SUMMARY (FOLD-1 VAL SPLIT)")
print("========================================================")
display(df_metrics)

# Plot Confusion Matrices Side-by-Side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm_vid = confusion_matrix(val_targets, val_video_base_preds)
cm_aud = confusion_matrix(val_targets, val_audio_base_preds)
cm_fus = confusion_matrix(val_targets, val_fusion_base_preds)

sns.heatmap(cm_vid, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Raw Video Base (dima806)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cm_aud, annot=True, fmt="d", cmap="Greens", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title("Raw Audio Base (MelodyMachine - Corrected)")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

sns.heatmap(cm_fus, annot=True, fmt="d", cmap="Purples", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[2])
axes[2].set_title("Raw Multimodal Base Ensemble")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("True")

plt.tight_layout()
plt.show()

## 9. Single Video Zero-Shot Inference Demo

In [ ]:
def predict_base_multimodal_video(video_path, w_video=0.5, w_audio=0.5):
    """
    Zero-shot inference helper for single video using raw pretrained base models.
    """
    print(f"🎬 Zero-shot inspecting video: {video_path}")
    p_vid = predict_video_base_probs(video_path)
    p_aud = predict_audio_base_probs(video_path)
    p_fusion = (w_video * p_vid) + (w_audio * p_aud)
    
    pred_idx = int(np.argmax(p_fusion))
    pred_label = CLASSES[pred_idx]
    conf = float(p_fusion[pred_idx])
    
    print("--- Base Model Zero-Shot Scores ---")
    print(f"📹 ViT Base (dima806):        {CLASSES[np.argmax(p_vid)]} ({np.max(p_vid)*100:.2f}%)")
    print(f"🎙️ Wav2Vec2 Base (MelodyMachine): {CLASSES[np.argmax(p_aud)]} ({np.max(p_aud)*100:.2f}%)")
    print(f"🤝 COMBINED ZERO-SHOT RESULT:   {pred_label} ({conf*100:.2f}% confidence)")
    return pred_label, conf

# Example Usage:
# predict_base_multimodal_video("/content/dataset_raw_val/AI/sample_video.mp4")